#29th Session

In [0]:
print("=====================Data Preparation========================")

data = [
    ("00000000", "06-26-2011", 200, "Exercise", "GymnasticsPro", "cash"),
    ("00000001", "05-26-2011", 300, "Exercise", "Weightlifting", "credit"),
    ("00000002", "06-01-2011", 100, "Exercise", "GymnasticsPro", "cash"),
    ("00000003", "06-05-2011", 100, "Gymnastics", "Rings", "credit"),
    ("00000004", "12-17-2011", 300, "Team Sports", "Field", "paytm"),
    ("00000005", "02-14-2011", 200, "Gymnastics", None, "cash")
]

df = spark.createDataFrame(data).toDF(
    "id", "tdate", "amt", "category", "product", "mode"
)
df.display()

=====================Data Preparation========================


id,tdate,amt,category,product,mode
00000000,06-26-2011,200,Exercise,GymnasticsPro,cash
00000001,05-26-2011,300,Exercise,Weightlifting,credit
00000002,06-01-2011,100,Exercise,GymnasticsPro,cash
00000003,06-05-2011,100,Gymnastics,Rings,credit
00000004,12-17-2011,300,Team Sports,Field,paytm
00000005,02-14-2011,200,Gymnastics,null,cash


In [0]:
from pyspark.sql.functions import *

df_withColumn = (
    df.withColumn("category", expr("upper(category)"))
    .withColumn("product", expr("product || '~ zeyo'"))
    .withColumn("amt", expr("amt + 100"))
    .withColumn("tdate", expr("split(tdate,'-')[2]"))
    .withColumn("id", expr("cast(id as int)"))
    .withColumn("status", expr("case when mode = 'cash' then 0 else 1 end"))
    .withColumnRenamed("tdate", "Year")
)
df_withColumn.display()

id,Year,amt,category,product,mode,status
0,2011,300,EXERCISE,GymnasticsPro~ zeyo,cash,0
1,2011,400,EXERCISE,Weightlifting~ zeyo,credit,1
2,2011,200,EXERCISE,GymnasticsPro~ zeyo,cash,0
3,2011,200,GYMNASTICS,Rings~ zeyo,credit,1
4,2011,400,TEAM SPORTS,Field~ zeyo,paytm,1
5,2011,300,GYMNASTICS,null,cash,0


In [0]:
# Define the data records
data = [
    ("Alice Smith", "New York", 1250),
    ("Bob Jones", "Los Angeles", 850),
    ("Julia Roberts", "San Jose", 2100),
    ("Alice Smith", "New York", 450),
    ("Bob Jones", "Los Angeles", 3120),
    ("Alice Smith", "New York", 675),
    ("Bob Jones", "Los Angeles", 850),
    ("Julia Roberts", "San Jose", 540),
    ("Julia Roberts", "San Jose", 540)
]

# Create the data frame
df = spark.createDataFrame(data).toDF("Name","City","Amount")

# Show the data frame contents
df.display()


Name,City,Amount
Alice Smith,New York,1250
Bob Jones,Los Angeles,850
Julia Roberts,San Jose,2100
Alice Smith,New York,450
Bob Jones,Los Angeles,3120
Alice Smith,New York,675
Bob Jones,Los Angeles,850
Julia Roberts,San Jose,540
Julia Roberts,San Jose,540


In [0]:

from pyspark.sql.functions import *

df_agg = df.groupBy("Name").agg(
    sum("Amount").alias("Total_amt"),
    count("Name").alias("Count"),
    collect_list("Amount").alias("Amount_list"),
    collect_set("Amount").alias("Amount_set")
)
df_agg.display()

Name,Total_amt,Count,Amount_list,Amount_set
Alice Smith,2375,3,"List(1250, 450, 675)","List(1250, 450, 675)"
Bob Jones,4820,3,"List(850, 850, 3120)","List(850, 3120)"
Julia Roberts,3180,3,"List(2100, 540, 540)","List(2100, 540)"


In [0]:
df_agg = df.groupBy("Name","City").agg(
    sum("Amount").alias("Total_amt"),
    count("Name").alias("Count"),
    collect_list("Amount").alias("Amount_list"),
    collect_set("Amount").alias("Amount_distinct")

)

df_agg.display()

Name,City,Total_amt,Count,Amount_list,Amount_distinct
Alice Smith,New York,2375,3,"List(1250, 450, 675)","List(1250, 450, 675)"
Bob Jones,Los Angeles,4820,3,"List(850, 850, 3120)","List(850, 3120)"
Julia Roberts,San Jose,3180,3,"List(2100, 540, 540)","List(2100, 540)"


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("EmployeeData").getOrCreate()

data = [
    (1, "Jhon", 4000),
    (2, "Tim David", 12000),
    (3, "Json Bhrendroff", 7000),
    (4, "Jordon", 8000),
    (5, "Green", 14000),
    (6, "Brewis", 6000),
]

columns = ["emp_id", "emp_name", "salary"]

df = spark.createDataFrame(data, schema=columns)
df.show()

+------+---------------+------+
|emp_id|       emp_name|salary|
+------+---------------+------+
|     1|           Jhon|  4000|
|     2|      Tim David| 12000|
|     3|Json Bhrendroff|  7000|
|     4|         Jordon|  8000|
|     5|          Green| 14000|
|     6|         Brewis|  6000|
+------+---------------+------+



In [0]:
from pyspark.sql.functions import *
df_grade= df.withColumn("grade", expr("case when salary > 10000 then 'A' when salary  > 5000 then 'B' else 'C' end"))
df_grade.display()

emp_id,emp_name,salary,grade
1,Jhon,4000,C
2,Tim David,12000,A
3,Json Bhrendroff,7000,B
4,Jordon,8000,B
5,Green,14000,A
6,Brewis,6000,B
